In [ ]:
import os
import re
import time
import json
import math
import random
import logging
from datetime import datetime
from urllib.parse import urljoin, urlparse

import requests
from bs4 import BeautifulSoup
import pandas as pd

# -------------------- Config --------------------
OUTPUT_PARQUET = "./lajornada_maya_articles.parquet"
REQUEST_TIMEOUT = 15
MAX_RETRIES = 3
SLEEP_BASE = (1.2, 2.4)  # polite delays (min, max) seconds
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/118.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "es-419,es;q=0.9,en;q=0.8",
}

BASE = "https://www.lajornadamaya.mx"
REGIONS = ["quintana-roo", "yucatan", "campeche", "nacional", "internacional"]
CATEGORIES = ["politica", "economia", "cultura", "turismo", "ecologia", "sociedad", "deportes"]

# Spanish month map for robust date parsing like "08 de Octubre del 2025"
SPANISH_MONTHS = {
    "enero": 1, "febrero": 2, "marzo": 3, "abril": 4, "mayo": 5, "junio": 6,
    "julio": 7, "agosto": 8, "septiembre": 9, "setiembre": 9, "octubre": 10,
    "noviembre": 11, "diciembre": 12,
}

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

# -------------------- Helpers --------------------
def sleep_polite():
    """Sleep a random short interval to avoid hammering the server."""
    time.sleep(random.uniform(*SLEEP_BASE))

def get(url, allow_404=False):
    """Robust GET with retries and polite delay."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            if resp.status_code == 404 and allow_404:
                return resp
            resp.raise_for_status()
            return resp
        except requests.RequestException as e:
            logging.warning(f"[{attempt}/{MAX_RETRIES}] GET failed {url} -> {e}")
            sleep_polite()
    return None

def text_clean(s):
    if not s:
        return None
    return re.sub(r"\s+", " ", s).strip()

def parse_spanish_date(s):
    """
    Try several Spanish date formats:
    - '08 de Octubre del 2025'
    - 'Miércoles. 13. de. Agosto. del. 2025'
    - '23/06/2025'
    Returns ISO date 'YYYY-MM-DD' or None.
    """
    if not s:
        return None
    s0 = s.lower()
    # dd/mm/yyyy
    m = re.search(r"\b(\d{1,2})/(\d{1,2})/(\d{4})\b", s0)
    if m:
        d, mth, y = map(int, m.groups())
        try:
            return datetime(y, mth, d).date().isoformat()
        except ValueError:
            pass
    # '13 de agosto del 2025' (allow dots and extra words)
    m = re.search(r"(\d{1,2}).*?de\s+([a-záéíóú]+).*?(?:del|de)\s+(\d{4})", s0)
    if m:
        d = int(re.sub(r"\D", "", m.group(1)))
        month_name = m.group(2).replace("á","a").replace("é","e").replace("í","i").replace("ó","o").replace("ú","u")
        y = int(m.group(3))
        mth = SPANISH_MONTHS.get(month_name, None)
        if mth:
            try:
                return datetime(y, mth, d).date().isoformat()
            except ValueError:
                pass
    return None

def is_article_url(url):
    """
    Article URLs observed like:
      https://www.lajornadamaya.mx/yucatan/237211/la-nacion-...
    We consider pattern '/{region}/{numeric_id}/...'
    """
    try:
        path = urlparse(url).path.rstrip("/")
        parts = [p for p in path.split("/") if p]
        if len(parts) >= 2 and parts[0] in REGIONS and re.fullmatch(r"\d+", parts[1]):
            return True
    except Exception:
        return False
    return False

def discover_next_link(soup, current_url):
    """
    Try to find a 'next page' link commonly used by infinite scroll plugins:
    - rel="next"
    - classes containing 'next' or 'more'
    If not found, return None (caller may try /page/N or ?page=N strategies).
    """
    # rel="next"
    a = soup.find("a", attrs={"rel": "next"})
    if a and a.get("href"):
        return urljoin(current_url, a["href"])

    # class contains "next" or "more"
    for a in soup.find_all("a", href=True):
        cls = " ".join(a.get("class", [])).lower()
        txt = (a.get_text() or "").lower()
        if "next" in cls or "siguiente" in cls or "more" in cls or "mas" in cls:
            return urljoin(current_url, a["href"])
        if "siguiente" in txt or "más" in txt or "mas" in txt or "next" in txt:
            return urljoin(current_url, a["href"])

    return None

# -------------------- Sitemaps strategy --------------------
def fetch_sitemap_urls():
    """
    Try common sitemap locations and gather article URLs.
    Filters to article-like URLs via is_article_url().
    """
    sitemap_candidates = [
        f"{BASE}/sitemap.xml",
        f"{BASE}/sitemap_index.xml",
        f"{BASE}/wp-sitemap.xml",
    ]
    urls = set()
    for sm in sitemap_candidates:
        resp = get(sm, allow_404=True)
        if not resp or resp.status_code != 200 or "xml" not in resp.headers.get("Content-Type","").lower():
            continue
        try:
            # basic XML parsing via regex to avoid extra deps
            locs = re.findall(r"<loc>(.*?)</loc>", resp.text)
            for loc in locs:
                loc = loc.strip()
                if loc.endswith(".xml"):
                    # sub-sitemap
                    sub = get(loc, allow_404=True)
                    if sub and sub.status_code == 200:
                        for u in re.findall(r"<loc>(.*?)</loc>", sub.text):
                            if is_article_url(u):
                                urls.add(u.split("?")[0])
                else:
                    if is_article_url(loc):
                        urls.add(loc.split("?")[0])
        except Exception as e:
            logging.warning(f"Sitemap parse failed for {sm}: {e}")
        sleep_polite()
    return urls

# -------------------- Category listing strategy --------------------
def enumerate_category_listing(region, category, hard_cap_pages=5000):
    """
    Attempt to traverse category listing to harvest article URLs without JS scrolling:
    1) Load the landing page.
    2) Extract links to article pages.
    3) Find an explicit 'next' link; if absent, try '?page=N' and '/page/N/'.
    Stops when no new links are found or after hard_cap_pages.
    """
    start = f"{BASE}/{region}/{category}"
    seen = set()
    harvested = set()
    q = [(start, 1, "auto")]  # (url, page_index, mode)

    while q:
        url, pidx, mode = q.pop(0)
        if (url, pidx, mode) in seen:
            continue
        seen.add((url, pidx, mode))

        resp = get(url)
        if not resp:
            continue
        soup = BeautifulSoup(resp.text, "html.parser")

        # Collect article links on this page
        for a in soup.find_all("a", href=True):
            link = urljoin(url, a["href"])
            if is_article_url(link):
                harvested.add(link.split("?")[0])

        # Try discover 'next' link in DOM
        nxt = discover_next_link(soup, url)
        if nxt and (nxt, pidx+1, "dom") not in seen:
            q.append((nxt, pidx+1, "dom"))

        # If no explicit next, try typical paginations (?page=N and /page/N/)
        if not nxt:
            for pattern in ("param", "path"):
                if pattern == "param":
                    trial = f"{start}?page={pidx+1}"
                else:
                    trial = f"{start}/page/{pidx+1}/"
                if (trial, pidx+1, pattern) not in seen:
                    # Stop if we hit too many pages without growth
                    if pidx+1 <= hard_cap_pages:
                        q.append((trial, pidx+1, pattern))

        sleep_polite()

        # Heuristic early stop: if the last few pages didn't add anything new
        if len(seen) > 20 and len(harvested) < 5:
            # category probably empty or blocked
            break

        if len(seen) > hard_cap_pages:
            break

    return harvested

# -------------------- Article parsing --------------------
def extract_topic_from_article(soup, region):
    """
    Find a link that looks like '/{region}/{category}' to label the topic.
    """
    pat = re.compile(rf"/{region}/({'|'.join(CATEGORIES)})/?$")
    for a in soup.find_all("a", href=True):
        m = pat.search(a["href"])
        if m:
            return m.group(1)
    # Sometimes breadcrumb text like "Quintana Roo > Cultura"
    crumb = soup.get_text(separator=" ", strip=True)
    for cat in CATEGORIES:
        if re.search(rf"\b{cat}\b", crumb, re.IGNORECASE):
            return cat
    return None

def extract_article_fields(url):
    """
    Parse title, subtitle (if any), author, date, city/region breadcrum, and main body text.
    """
    resp = get(url)
    if not resp:
        return None
    soup = BeautifulSoup(resp.text, "html.parser")

    # Title/subtitle
    title = None
    for sel in ["h1", "h1.entry-title", "h1.post-title", "header h1"]:
        h = soup.select_one(sel)
        if h:
            title = text_clean(h.get_text())
            break

    subtitle = None
    for sel in ["h2", ".subtitulo", ".dek", ".entry-subtitle"]:
        h2 = soup.select_one(sel)
        if h2:
            t = text_clean(h2.get_text())
            if t and (not title or t != title):
                subtitle = t
                break

    # Date
    date_iso = None
    # 1) <time datetime="">
    t = soup.find("time")
    if t and (t.get("datetime") or t.get_text()):
        cand = t.get("datetime") or t.get_text()
        date_iso = parse_spanish_date(cand) or text_clean(cand)
    # 2) date blocks common in news
    if not date_iso:
        for sel in [".fecha", ".date", ".published", ".post-created", ".entry-date"]:
            d = soup.select_one(sel)
            if d:
                date_iso = parse_spanish_date(d.get_text())
                if date_iso:
                    break

    # Region from URL
    try:
        parts = [p for p in urlparse(url).path.split("/") if p]
        region = parts[0] if parts else None
    except Exception:
        region = None

    # Topic from breadcrumb/links
    topic = extract_topic_from_article(soup, region) if region else None

    # Author (best-effort)
    author = None
    for sel in [".author", ".byline", ".post-author", ".autor"]:
        a = soup.select_one(sel)
        if a:
            author = text_clean(a.get_text())
            break

    # Main text (collect paragraphs in plausible containers; fallback to article p)
    body_candidates = [
        'article',
        'div[itemprop="articleBody"]',
        '.entry-content',
        '.post-content',
        '.field--name-body',
        '.contenido',
        '.nota',
    ]
    paragraphs = []
    for sel in body_candidates:
        for container in soup.select(sel):
            ps = container.find_all("p")
            if ps and len(ps) >= 2:
                for p in ps:
                    txt = text_clean(p.get_text(" "))
                    if txt:
                        paragraphs.append(txt)
                break
        if paragraphs:
            break
    if not paragraphs:
        # Fallback: any <article> p or global p
        for p in soup.find_all("p"):
            txt = text_clean(p.get_text(" "))
            if txt:
                paragraphs.append(txt)

    main_text = text_clean(" ".join(paragraphs)) if paragraphs else None

    # Housekeeping
    if main_text:
        # Remove common ad/promo lines if any
        main_text = main_text.replace(
            "Conoce los servicios publicitarios que impulsarán tu marca a otro nivel.", ""
        ).strip()

    return {
        "url": url,
        "title": title,
        "sub_title": subtitle,
        "author": author,
        "date": date_iso,     # ISO 'YYYY-MM-DD' when parsed
        "topic": topic,       # política/economía/...
        "region": region,     # yucatan/quintana-roo/...
        "main_text": main_text,
        "scraped_at": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    }

# -------------------- Storage --------------------
def save_parquet(rows, path=OUTPUT_PARQUET):
    """Append-dedupe write to a single Parquet file."""
    if not rows:
        return
    df_new = pd.DataFrame(rows)
    if os.path.exists(path):
        df_old = pd.read_parquet(path)
        df = pd.concat([df_old, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)
    else:
        os.makedirs(os.path.dirname(path), exist_ok=True) if os.path.dirname(path) else None
        df = df_new.drop_duplicates(subset=["url"]).reset_index(drop=True)
    df.to_parquet(path, engine="pyarrow", compression="gzip")
    logging.info(f"Saved {len(df_new)} new rows | total unique: {len(df)} -> {path}")

# -------------------- Orchestrator --------------------
def main():
    harvested_urls = set()

    # 1) Try sitemaps for the exhaustive set
    logging.info("Attempting sitemap discovery...")
    sm_urls = fetch_sitemap_urls()
    logging.info(f"Sitemap URLs found (article-like): {len(sm_urls)}")
    harvested_urls |= sm_urls

    # 2) Fallback: enumerate each {region}/{category}
    for region in REGIONS:
        for category in CATEGORIES:
            listing = f"{BASE}/{region}/{category}"
            logging.info(f"Enumerating listing: {listing}")
            urls_cat = enumerate_category_listing(region, category)
            logging.info(f"Found {len(urls_cat)} article URLs in {region}/{category}")
            harvested_urls |= urls_cat

    # 3) Parse articles incrementally in batches
    logging.info(f"Total unique URLs to parse: {len(harvested_urls)}")
    parsed_batch = []
    for i, url in enumerate(sorted(harvested_urls)):
        art = extract_article_fields(url)
        if art:
            parsed_batch.append(art)

        # Write in small batches to be resilient
        if len(parsed_batch) >= 50:
            save_parquet(parsed_batch, OUTPUT_PARQUET)
            parsed_batch = []

        # Be polite
        sleep_polite()

    # Flush remaining
    save_parquet(parsed_batch, OUTPUT_PARQUET)
    logging.info("Done.")

if __name__ == "__main__":
    main()
